# Trigger pt studies

The goal of this notebook is to study the pt requirements to put us in the plateau of the trigger efficiency. This is slightly tricky because for the 4mu channel, we expect that one muon from each LJ will pass the trigger whereas for the 2mu2e channel, both muons from one dark photon decay will have to pass the trigger.

## Setup

In [ ]:
# python
import sys
import importlib
# columnar analysis
from coffea import processor
# local
sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import sidm_processor, llpnanoaodschema, utilities
importlib.reload(sidm_processor)
importlib.reload(llpnanoaodschema)
importlib.reload(utilities)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
import numpy as np

In [ ]:
samples_2mu2e = [
    '2Mu2E_200GeV_0p25GeV_10p0mm',
    '2Mu2E_200GeV_0p25GeV_0p01mm',
    '2Mu2E_200GeV_5p0GeV_2p0mm',
    '2Mu2E_200GeV_5p0GeV_200p0mm',
]
fileset_2mu2e = utilities.make_fileset(samples_2mu2e, "llpNanoAOD_v2", max_files=1, location_cfg="signal_2mu2e_v10.yaml")

samples_4mu = [
    '4Mu_200GeV_5p0GeV_200p0mm',
]
fileset_4mu = utilities.make_fileset(samples_4mu, "llpNanoAOD_v2", max_files=1, location_cfg="signal_4mu_v10.yaml")

samples = samples_2mu2e + samples_4mu
fileset = fileset_2mu2e | fileset_4mu

In [ ]:
runner = processor.Runner(
    executor=processor.IterativeExecutor(),
    #executor=processor.FuturesExecutor(),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    maxchunks=1,
    skipbadfiles=True
)

channels = [
    "baseNoLj",
    "baseNoLj_noTrigger",
]
p = sidm_processor.SidmProcessor(
    channels,
    ["muon_base", "dsaMuon_base", "lepton_genA_base", "trig_base"],
    #verbose=True,
    unweighted_hist=True,
)

output = runner.run(fileset, treename='Events', processor_instance=p)
out = output["out"]

## 2D plots

Plotting the numerator, denominator, and efficiency plots for each sample.

The denominator has the lepton-jet source cuts and PV requirements applied (but no lepton-jet level cuts). The numerator has the same cuts applied, but additionally requires the event to pass the trigger.

In [ ]:
#Define custom color map
import matplotlib.colors as mcolors
base_cmap = plt.get_cmap("viridis")
cmap_colors = base_cmap(np.linspace(0, 1, 256))

# Force the very bottom of the scale to pure white (RGBA)
cmap_colors[0] = [1.0, 1.0, 1.0, 1.0]
custom_cmap = mcolors.ListedColormap(cmap_colors)

#Define grid
ns = len(samples)
nc = len(channels)

first = True

num_ch = channels[0]
denom_ch = channels[1]

plot_name = "all_muon0_pt_vs_all_muon1_pt"

#Plot one row at a time (to avoid weird scaling effects)
for sample in samples:
    fig, axes = plt.subplots(1, nc+1, figsize=(nc * 15, 10)) 

    for i, ch in enumerate(channels):
        plt.subplot(1, nc+1, i+1)
          
        utilities.plot(out[sample]["hists"][plot_name][ch, :,:],cmap=custom_cmap)

        ax = axes[i]
        

        ax.text(0.95, 0.95, sample, 
                transform=ax.transAxes, 
                fontsize=25, 
                color='black',
                ha='right', 
                va='bottom')
        if first:
            ax.text(0.8, 1.02, ch, 
                transform=ax.transAxes, 
                fontsize=25, 
                color='red',
                ha='right', 
                va='bottom')
    i=i+1
    plt.subplot(1, nc+1, i+1)

    num  =  out[sample]["hists"][plot_name][num_ch,   :,:]
    denom = out[sample]["hists"][plot_name][denom_ch, :,:]

    ratio = num.copy()
    ratio.view().value = np.divide(num.values(), denom.values(), out=-1*np.ones_like(num.values()), where=denom.values()!=0)
    ratio.view().value[ratio.values() > 1.0] = np.nan

    ratio.plot2d(cmap=custom_cmap)
    #utilities.plot(ratio,cmap=custom_cmap)
    ax = plt.gca()
    ax.set_xlim([0,98])
    ax.set_ylim([0,98])

    first = False
        


Okay... so far, these plots are telling me that we hit the plateau of the trigger efficiency when the sub-leading muon pt is about 25 GeV. It doesn't seem to depend too heavily on the leading event muon pt. I should run these again to zoom in on the 20-30 GeV region to make a more specific pt threshold recommendation. 

## 1D plots

Now let's look at the 1D efficiency vs leading or sub-leading muon pt. 

In [ ]:
#Define a function to plot a row of ratio plots
def plot_ratio_row(nums, dens, sample,**kwargs):
    """
    Plots n ratio plots in a single row.
    nums: list of numerators (each can be a hist or a list of hists)
    dens: list of denominators (hists)
    """
    n = len(dens)
    
    # Lower-panel label/limits default to the efficiency convention but can be
    # overridden for a generic ratio; popped so they do not reach hep.histplot.
    ratio_ylabel = kwargs.pop("ratio_ylabel", "Efficiency")
    ratio_ylim = kwargs.pop("ratio_ylim", (0, 1.2))
    
    # squeeze=False guarantees axes is always shape (2, n)
    # sharex='col' shares the x-axis vertically but not horizontally
    fig, axes = plt.subplots(
        2, n, figsize=(8 * n, 12), sharex='col',
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0},
        squeeze=False
    )

    for i in range(n):
        ax1 = axes[0, i]
        ax2 = axes[1, i]
        
        den = dens[i]
        num = nums[i]
        
        plt.sca(ax1)
        sam = sample
        
        # Handle kwargs that might be passed as lists (one for each column)
        label = None
        if "legend" in kwargs:
            label = kwargs["legend"][i] if isinstance(kwargs["legend"], list) else kwargs["legend"][0]

        utilities.plot(den, flow='none', color="k", skip_label=True, label=label)

        if not isinstance(num, list):
            num = [num]

        for x in num:
            utilities.plot(x, flow='none', label=label)

        if "legend" in kwargs:
            title = kwargs["text"][i] if "text" in kwargs and isinstance(kwargs["text"], list) else kwargs.get("text")
            ax1.legend(title=title, alignment="left")

        if "ylim" in kwargs:
            ax1.set_ylim(kwargs["ylim"])

        if i ==0:
            ax1.text(95, 25, sam, 
#                transform=ax.transAxes, 
                fontsize=25, 
                color='black',
                ha='right', 
                va='bottom')
            
        # Only draw the y-axis label on the leftmost plot to keep the row clean
        if "ylabel" in kwargs and i == 0:
            ax1.set_ylabel(kwargs["ylabel"])


        plt.sca(ax2)
        for x in num:
            try:
                eff, errors = utilities.get_eff_hist(x, den)
            except ValueError:
                # num is not a subset of den (a shape ratio, not an efficiency): the
                # binomial Clopper-Pearson interval is invalid and ratio_uncertainty
                # raises -- fall back to a Gaussian-propagated ratio.
                eff, errors = _gaussian_ratio(x, den)
            utilities.plot(eff, histtype='errorbar', yerr=errors, skip_label=True)

        # Only draw the ratio y-axis label on the leftmost plot
        if i == 0:
            ax2.set_ylabel(ratio_ylabel)
            
        if ratio_ylim is not None:
            ax2.set_ylim(*ratio_ylim)

        if "xlabel" in kwargs:
            # Handle if they pass a list of xlabels (one per column) or just one string
            xlabel = kwargs["xlabel"][i] if isinstance(kwargs["xlabel"], list) else kwargs["xlabel"]
            ax2.set_xlabel(xlabel)

    plt.tight_layout()
    return fig, axes

In [ ]:
pt_cut = 24
for sample in samples:
    nums = [out[sample]["hists"][plot_name][ num_ch, sum,:],out[sample]["hists"][plot_name][ num_ch, :,sum], out[sample]["hists"][plot_name][ num_ch, :,hist.loc(pt_cut)::sum]]
    denoms = [out[sample]["hists"][plot_name][denom_ch, sum,:],out[sample]["hists"][plot_name][denom_ch, :,sum],out[sample]["hists"][plot_name][ denom_ch, :,hist.loc(pt_cut)::sum]]
    plot_ratio_row(nums,denoms,sample,xlabel=["Sub-leading muon pT [GeV]","Leading muon pT [GeV]",f"Leading muon pT if sub-leading > {pt_cut}"])


From the first plot: the overall efficiency (plateau value) is different for each sample, but they each seem to hit the plateau when the sub-leading muon is approximately 25 GeV. 

From the second plot: these plots are inclusive of sub-leading muon pt. There's not a clean turn on curve or an obvious ut-off for leading muon pt here

From the third plot: Now I am plotting the trigger efficiency vs leading muon pt only in events where the sub-leading muon pt passes our pt cut (24 GeV). By my eye, those distributions are pretty flat. I think just a cut on the sub-leading muon pt is enough to put us in the plateau of the trigger efficiency. I don't think we need an additional cut on the leading muon pt (which by definition will be higher than 24 or whatever anyway)

This will have a different effect on the two samples... for the 2mu2e sample, we are requiring that *both* of the muons from a single dark photon are above 24 GeV. For the 4Mu sample, we are requiring that either A) *both* of the muons from a single dark photon are above 24 GeV or B) each dark photon produces at least one muon with pt > 24 GeV. The next step will be to figure out how to apply this event-level cut in our framework.

Or... if there aren't many cases of scenario A, then the cut could be
- For 2mu2e, require two muons in one LJ with pt > 24
- For 4mu, require each LJ to have at least one muon with pt > 24

That brings it to a LJ level requirement instead.

I'm also curious how this effects the separate LJ pt > 30 GeV cut that we have. That might not be needed at all for muon LJs now